In [1]:
import pandas as pd
import re
import numpy as np

In [2]:
completions_raw = pd.read_csv(
    r"..\data\raw\c2024_a.csv",
    dtype=str, 
    low_memory=False
)

print(completions_raw.shape)
completions_raw.head()

(307707, 64)


,UNITID,CIPCODE,MAJORNUM,AWLEVEL,XCTOTALT,CTOTALT,XCTOTALM,CTOTALM,XCTOTALW,CTOTALW,...,XCUNKNM,CUNKNM,XCUNKNW,CUNKNW,XCNRALT,CNRALT,XCNRALM,CNRALM,XCNRALW,CNRALW
0,100654,01.0999,1,5,R,12,R,2,R,10,...,Z,0,Z,0,Z,0,Z,0,Z,0
1,100654,01.1001,1,5,R,10,R,1,R,9,...,Z,0,Z,0,Z,0,Z,0,Z,0
2,100654,01.1001,1,7,R,7,R,1,R,6,...,Z,0,Z,0,R,2,R,1,R,1
3,100654,01.1001,1,17,R,6,R,3,R,3,...,R,0,R,2,R,3,R,2,R,1
4,100654,01.9999,1,5,R,2,R,1,R,1,...,Z,0,Z,0,Z,0,Z,0,Z,0


In [3]:
# Define the columns
cols = [
    "UNITID",
    "CIPCODE",        # cip6 format (xx.xxxx)
    "AWLEVEL",        # credential level
    "CTOTALT",        # total completions
    "CTOTALM",        # male
    "CTOTALW"        # female
]

completions = completions_raw[cols].copy()

In [4]:
# Unitid
completions["unitid_n"] = (
    completions["UNITID"]
    .astype(str)
    .str.strip()
)

In [5]:
# Build cip4
completions['CIPCODE'] = completions['CIPCODE'].astype(str)

# Remove decimal, pad to 6 digits, and slice
completions['cip6'] = (
    completions['CIPCODE']
    .str.replace('.', '', regex=False)
    .str.zfill(6)
    .str[:6]
)

# First four digits
completions['cip4'] = completions['cip6'].str[:4]

In [6]:
# Drop cip 99 (a placeholder for programs that cannot be traditionally coded)
completions = completions[completions["cip4"] != "99"]

In [7]:
# AWLEVEL to credlev_n
completions['credlev_n'] = pd.to_numeric(completions["AWLEVEL"], errors="coerce").astype("Int64")

In [8]:
# Completions counts
for c in ["CTOTALT", "CTOTALM", "CTOTALW"]:
    completions[c] = pd.to_numeric(completions[c], errors="coerce").fillna(0).astype(int)

completions = completions.rename(columns={
    "CTOTALT": "completions",
    "CTOTALM": "completions_men",
    "CTOTALW": "completions_women"
})

In [9]:
# Add a year column
completions["year"] = 2024

In [10]:
completions['AWLEVEL'] = completions['AWLEVEL'].astype(int)

In [11]:
completions['AWLEVEL'].dtype

dtype('int64')

In [12]:
completions['AWLEVEL'].head()

0     5
1     5
2     7
3    17
4     5
Name: AWLEVEL, dtype: int64

In [13]:
completions['AWLEVEL'].value_counts().sort_index()

AWLEVEL
2      29688
3      48741
4       2445
5     104090
6      14936
7      45516
8       5378
17     13469
18      3171
19       372
20      4543
21     35358
Name: count, dtype: int64

In [14]:
# Drop unused degree types
valid_awlevels = [2, 3, 4, 5]
completions = completions[completions['AWLEVEL'].isin(valid_awlevels)].copy()

In [15]:
# Create a dictionary for the credlev codes, limiting it to the ones I will be using
credlev_map = {
    1: [2, 4],  # IPEDS codes for certificates
    2: [3],     # IPEDS codes for associates
    3: [5]         # IPEDS codes for bachelors
}

def map_credlevel(awlevel):
    for proj_level, ipeds_codes in credlev_map.items():
        if awlevel in ipeds_codes:
            return proj_level
    return np.nan

completions["credlev_proj"] = completions["AWLEVEL"].apply(map_credlevel)

In [16]:
completions.head(1)

,UNITID,CIPCODE,AWLEVEL,completions,completions_men,completions_women,unitid_n,cip6,cip4,credlev_n,year,credlev_proj
0,100654,01.0999,5,12,2,10,100654,010999,0109,5,2024,3


In [17]:
completions.info()

<class 'pandas.core.frame.DataFrame'>
Index: 184964 entries, 0 to 307706
Data columns (total 12 columns):
 #   Column             Non-Null Count   Dtype 
---  ------             --------------   ----- 
 0   UNITID             184964 non-null  object
 1   CIPCODE            184964 non-null  object
 2   AWLEVEL            184964 non-null  int64 
 3   completions        184964 non-null  int64 
 4   completions_men    184964 non-null  int64 
 5   completions_women  184964 non-null  int64 
 6   unitid_n           184964 non-null  object
 7   cip6               184964 non-null  object
 8   cip4               184964 non-null  object
 9   credlev_n          184964 non-null  Int64 
 10  year               184964 non-null  int64 
 11  credlev_proj       184964 non-null  int64 
dtypes: Int64(1), int64(6), object(5)
memory usage: 18.5+ MB


In [18]:
# Verifiy data accuracy
completions[['CIPCODE', 'cip6', 'cip4']].drop_duplicates().head(20)

# how many unique values at each level
completions['CIPCODE'].nunique(), completions['cip6'].nunique(), completions['cip4'].nunique()

(1451, 1451, 394)

In [19]:
# Exploring 
cip_totals_raw = (
    completions
    .groupby('CIPCODE', as_index=False)['completions']
    .sum()
    .sort_values('completions', ascending=False)
)

cip_totals_raw.head(20)

,CIPCODE,completions
1450,99,3589805
600,24.0101,332625
1321,51.3801,233786
1344,52.0201,201838
871,42.0101,137848
601,24.0102,123819
608,26.0101,84466
209,11.0701,51835
229,12.0401,50921
1416,52.1401,46005


In [20]:
# Exploring 
cip6_totals = (
    completions
    .groupby('cip6', as_index=False)['completions']
    .sum()
    .sort_values('completions', ascending=False)
)

cip4_totals = (
    completions
    .groupby('cip4', as_index=False)['completions']
    .sum()
    .sort_values('completions', ascending=False)
)

In [21]:
# Exploring 
mask_51 = completions['CIPCODE'].astype(str).str.startswith('51')

cip51_raw = (
    completions.loc[mask_51]
    .groupby('CIPCODE', as_index=False)['completions']
    .sum()
    .sort_values('completions', ascending=False)
)

cip51_raw.head(20)
cip51_raw['completions'].sum()

np.int64(585093)

In [22]:
# Exploring
completions['completions'].sum(), cip6_totals['completions'].sum(), cip4_totals['completions'].sum()

(np.int64(7179610), np.int64(7179610), np.int64(7179610))

In [23]:
# Exloring
cip51_cip6 = (
    completions.loc[mask_51]
    .groupby('cip6', as_index=False)['completions']
    .sum()
    .sort_values('completions', ascending=False)
)

cip51_cip4 = (
    completions.loc[mask_51]
    .groupby('cip4', as_index=False)['completions']
    .sum()
    .sort_values('completions', ascending=False)
)

In [24]:
# Exploring
(
    completions.loc[mask_51]
    .groupby('credlev_n', as_index=False)['completions']
    .sum()
    .sort_values('completions', ascending=False)
)

# example: health by institution and award level
(
    completions.loc[mask_51]
    .groupby(['unitid_n', 'credlev_n'], as_index=False)['completions']
    .sum()
    .sort_values('completions', ascending=False)
    .head(3)
)

,unitid_n,credlev_n,completions
4467,413413,5,8155
4981,454227,5,7843
4709,441371,3,7716


In [25]:
cip51_raw.head(1)

,CIPCODE,completions
176,51.3801,233786


In [26]:
cip51_raw['completions'].sum()

np.int64(585093)

In [27]:
# Exploring 
(
    completions[completions['cip4'] == '5138']
    .groupby('credlev_n', as_index=False)['completions']
    .sum()
)

,credlev_n,completions
0,2,1395
1,3,85232
2,4,1268
3,5,153340


In [28]:
# Name credlev_proj descriptions
cred_desc_map = {
    1: "Certificate",
    2: "Assoicate",
    3: "Bachelor"
}

completions["cred_desc"] = completions["credlev_proj"].map(cred_desc_map)

In [29]:
# Create composite key
completions["composite_key"] = (
    completions["unitid_n"] + "|" +
    completions["cip4"] + "|" +
    completions["credlev_proj"].astype(str)
)

In [30]:
# uniqueness
completions["composite_key"].is_unique

False

In [31]:
# null checks
completions.isna().sum()

UNITID               0
CIPCODE              0
AWLEVEL              0
completions          0
completions_men      0
completions_women    0
unitid_n             0
cip6                 0
cip4                 0
credlev_n            0
year                 0
credlev_proj         0
cred_desc            0
composite_key        0
dtype: int64

In [32]:
# Verify that null values are ok
completions['credlev_proj'].value_counts(dropna=False)

credlev_proj
3    104090
2     48741
1     32133
Name: count, dtype: int64

In [33]:
completions.head(3)

,UNITID,CIPCODE,AWLEVEL,completions,completions_men,completions_women,unitid_n,cip6,cip4,credlev_n,year,credlev_proj,cred_desc,composite_key
0,100654,01.0999,5,12,2,10,100654,010999,0109,5,2024,3,Bachelor,100654|0109|3
1,100654,01.1001,5,10,1,9,100654,011001,0110,5,2024,3,Bachelor,100654|0110|3
4,100654,01.9999,5,2,1,1,100654,019999,0199,5,2024,3,Bachelor,100654|0199|3
